# VadCLIP voi Temporal CNN classifier
Thay classifier logits1 bang CNN, giu LGT-Adapter va alignment. Tat ca variant train from scratch; test chi dung de danh gia epoch 10. Chay cac cell tu tren xuong duoi.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
BASELINE_MODEL = SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
OUTPUT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_vadclip_seed234_repeat'
TRAIN_SEED = 234
PREVIOUS_RUN_TAG = 'cnn_vadclip_v1'

if not torch.cuda.is_available():
    raise RuntimeError('Hay chon Colab GPU runtime truoc khi chay notebook.')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Can co /content/UCFClipFeatures hoac UCFClipFeatures.zip/.tar/.tar.gz tren Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
for required in (SRC_DIR / 'ucf_train_cnn_vadclip.py', SRC_DIR / 'ucf_evaluate_cnn_vadclip.py',
                 SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_cnn_common.py', SRC_DIR / 'test_cnn_vadclip.py',
                 LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
                 LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy', FEATURE_ROOT):
    if not required.exists():
        raise FileNotFoundError(required)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Baseline tu train:', BASELINE_MODEL, 'FOUND' if BASELINE_MODEL.is_file() else 'MISSING')


## Smoke test
Kiem tra hai kieu scoring, shape, loss, backward va CLIP frozen truoc khi train that.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
subprocess.run([sys.executable, '-m', 'unittest', 'test_cnn_vadclip'], cwd=SRC_DIR, check=True)
from model_cnn_vadclip import CLIPVADCNN
from ucf_cnn_common import UCFCNNFeatureDataset, aggregate_video_logits, alignment_mil_loss, text_regularization_loss

test_data = UCFCNNFeatureDataset(FEATURE_ROOT, LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', 256)
assert len(test_data) == 290
for predicate in (lambda n: n < 256, lambda n: n > 256):
    found = False
    for index in range(len(test_data)):
        chunks, _, length, video_id, chunk_lengths = test_data[index]
        if predicate(length):
            assert chunks.shape[1:] == (256, 512)
            assert int(chunk_lengths.sum()) == length
            print('Chunk test PASS:', video_id, length, tuple(chunks.shape))
            found = True
            break
    assert found, 'Missing short or long test sequence'

device = torch.device('cuda')
visual = torch.randn(2, 256, 512, device=device)
lengths = torch.tensor([170, 256], device=device)
labels = torch.zeros(2, 14, device=device)
labels[0, 0] = 1
labels[1, 6] = 1
for mode in ('scoring_head', 'kernel_response'):
    model = CLIPVADCNN(14, 512, 256, 512, 1, 2, 8, 10, 10, device, kernel_size=3, score_mode=mode).to(device)
    text, logits1, logits2 = model(visual, None, list(__import__('ucf_cnn_common').TRAIN_LABEL_MAP.values()), lengths)
    assert logits1.shape == (2, 256, 1) and logits2.shape == (2, 256, 14)
    assert torch.equal(logits1[0, 170:], torch.zeros_like(logits1[0, 170:]))
    video_logits = aggregate_video_logits(logits1, lengths, 'topk')
    loss = torch.nn.functional.binary_cross_entropy_with_logits(video_logits, torch.tensor([0., 1.], device=device))
    loss = loss + alignment_mil_loss(logits2, labels, lengths) + text_regularization_loss(text)
    assert torch.isfinite(loss)
    loss.backward()
    assert model.cnn_classifier.projection[0].weight.grad is not None
    assert any(p.grad is not None for p in model.temporal.parameters())
    assert all(p.grad is None for p in model.clipmodel.parameters())
    print(mode, 'PASS', tuple(logits1.shape), tuple(logits2.shape), float(loss))
    del model
torch.cuda.empty_cache()


## Train va evaluate
Chay lai mot luot ca nam kernel 1, 3, 5, 7, 9 voi cung seed 234 nhu luot dau; chi dung scoring_head va Top-K de so sanh cong bang. Ket qua duoc luu rieng, khong ghi de luot dau. Baseline la model tu train, khong dung model cua tac gia.

In [ ]:
VARIANTS = {
    'k1_head_topk': (1, 'scoring_head', 'topk', 1.0, 1.0),
    'k3_head_topk': (3, 'scoring_head', 'topk', 1.0, 1.0),
    'k5_head_topk': (5, 'scoring_head', 'topk', 1.0, 1.0),
    'k7_head_topk': (7, 'scoring_head', 'topk', 1.0, 1.0),
    'k9_head_topk': (9, 'scoring_head', 'topk', 1.0, 1.0),
    'k3_kernel_topk': (3, 'kernel_response', 'topk', 1.0, 1.0),
    'k5_kernel_topk': (5, 'kernel_response', 'topk', 1.0, 1.0),
    'k3_head_max': (3, 'scoring_head', 'max', 1.0, 1.0),
    'k3_head_max_no_aux': (3, 'scoring_head', 'max', 0.0, 0.0),
}
SELECTED_VARIANTS = [f'k{k}_head_topk' for k in (1, 3, 5, 7, 9)]

def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Child process failed; traceback is printed above.')

def evaluate_command(model_type, model_path, eval_dir, analyze_dir, kernel=3, mode='scoring_head', train_config=None):
    command = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
            '--model-type', model_type, '--model-path', str(model_path),
            '--feature-root', str(FEATURE_ROOT),
            '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
            '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
            '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
            '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
            '--kernel-size', str(kernel), '--score-mode', mode,
            '--evaluate-dir', str(eval_dir), '--analyze-dir', str(analyze_dir)]
    if train_config is not None:
        command += ['--train-config-json', str(train_config)]
    return command

if not BASELINE_MODEL.is_file():
    raise FileNotFoundError(f'Can checkpoint baseline TU TRAIN tai {BASELINE_MODEL}; khong thay bang model_ucf.pth cua tac gia.')
baseline_eval = OUTPUT_ROOT / 'evaluate' / RUN_TAG / 'baseline_self_trained'
baseline_analyze = OUTPUT_ROOT / 'analyze' / RUN_TAG / 'baseline_self_trained'
baseline_metrics_path = baseline_eval / 'metrics.json'
if baseline_metrics_path.is_file():
    if json.loads(baseline_metrics_path.read_text())['model_path'] != str(BASELINE_MODEL):
        raise RuntimeError('Cached baseline metrics use another checkpoint. Change RUN_TAG.')
else:
    run_command(evaluate_command('baseline', BASELINE_MODEL, baseline_eval, baseline_analyze))

for name in SELECTED_VARIANTS:
    kernel, mode, pooling, align_weight, text_weight = VARIANTS[name]
    model_path = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_vadclip' / RUN_TAG / name / 'epoch_010.pth'
    train_dir = OUTPUT_ROOT / 'train' / RUN_TAG / name
    eval_dir = OUTPUT_ROOT / 'evaluate' / RUN_TAG / name
    analyze_dir = OUTPUT_ROOT / 'analyze' / RUN_TAG / name
    config_path = train_dir / 'config.json'
    if model_path.is_file():
        if not config_path.is_file():
            raise RuntimeError(f'Checkpoint exists without config: {model_path}. Change RUN_TAG.')
        saved = json.loads(config_path.read_text())
        expected = {'kernel_size': kernel, 'score_mode': mode, 'aggregation': pooling,
                    'lambda_alignment': align_weight, 'lambda_text': text_weight, 'seed': TRAIN_SEED,
                    'batch_size': 64, 'lr': 1e-5, 'max_epoch': 10,
                    'visual_length': 256, 'scheduler_milestones': [4, 8],
                    'feature_root': str(FEATURE_ROOT),
                    'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
        if any(saved.get(key) != value for key, value in expected.items()):
            raise RuntimeError(f'Checkpoint config mismatch for {name}. Change RUN_TAG.')
        print('Existing matching epoch-10 checkpoint; skip retraining:', model_path)
    else:
        command = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                   '--feature-root', str(FEATURE_ROOT),
                   '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
                   '--kernel-size', str(kernel), '--score-mode', mode, '--aggregation', pooling,
                   '--lambda-alignment', str(align_weight), '--lambda-text', str(text_weight),
                   '--seed', str(TRAIN_SEED), '--batch-size', '64', '--lr', '1e-5', '--max-epoch', '10',
                   '--scheduler-milestones', '4', '8', '--num-workers', '2',
                   '--epoch-checkpoint-dir', str(model_path.parent),
                   '--output-model-path', str(SRC_DIR / 'model' / 'cnn_vadclip' / RUN_TAG / f'{name}.pth'),
                   '--results-dir', str(train_dir)]
        run_command(command)
    metrics_path = eval_dir / 'metrics.json'
    if metrics_path.is_file():
        if json.loads(metrics_path.read_text())['model_path'] != str(model_path):
            raise RuntimeError(f'Cached metrics use another checkpoint for {name}. Change RUN_TAG.')
    else:
        run_command(evaluate_command('cnn', model_path, eval_dir, analyze_dir, kernel, mode, train_dir / 'config.json'))
    print('Completed:', name, json.loads((eval_dir / 'metrics.json').read_text())['classifier_auc'])


## So sanh ket qua epoch 10
AUC/AP la frame-level tren toan bo test set; Top-15 analysis chi dung ground truth sau train. Bang cuoi so sanh hai seed neu ket qua cu van con tren Drive.

In [ ]:
baseline = json.loads((baseline_eval / 'metrics.json').read_text())
print('Variant | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | avg_mAP | Top15 anomaly fraction')
for name in ['baseline_self_trained'] + SELECTED_VARIANTS:
    metrics = baseline if name == 'baseline_self_trained' else json.loads((OUTPUT_ROOT / 'evaluate' / RUN_TAG / name / 'metrics.json').read_text())
    diagnostic = json.loads((OUTPUT_ROOT / 'analyze' / RUN_TAG / name / 'summary.json').read_text())
    auc2 = f"{metrics['alignment_auc']:.2f}" if metrics['alignment_trained'] else 'N/A'
    ap2 = f"{metrics['alignment_ap']:.2f}" if metrics['alignment_trained'] else 'N/A'
    mean_ap = f"{metrics['avg_mAP']:.2f}" if metrics['alignment_trained'] else 'N/A'
    print(f"{name} | {metrics['classifier_auc']:.2f} | {metrics['classifier_ap']:.2f} | "
          f"{auc2} | {ap2} | {metrics['ano_auc']:.2f} | {mean_ap} | "
          f"{diagnostic['selected_anomaly_fraction_abnormal_video_macro']:.3f}")

print('\nSo sanh hai lan chay cung seed 234 (epoch 10):')
print('Kernel | AUC1 cu | AUC1 moi | Delta AUC1 | Ano-AUC cu | Ano-AUC moi | mAP cu | mAP moi')
for name in SELECTED_VARIANTS:
    old_path = OUTPUT_ROOT / 'evaluate' / PREVIOUS_RUN_TAG / name / 'metrics.json'
    new_path = OUTPUT_ROOT / 'evaluate' / RUN_TAG / name / 'metrics.json'
    if not old_path.is_file():
        print(f'{name}: khong tim thay metrics cu tai {old_path}')
        continue
    old = json.loads(old_path.read_text())
    new = json.loads(new_path.read_text())
    print(f"{name} | {old['classifier_auc']:.2f} | {new['classifier_auc']:.2f} | "
          f"{new['classifier_auc'] - old['classifier_auc']:+.2f} | "
          f"{old['ano_auc']:.2f} | {new['ano_auc']:.2f} | "
          f"{old['avg_mAP']:.2f} | {new['avg_mAP']:.2f}")
